# 03 · Partition-Aware Windowing (core contribution)
Add sliding-window temporal features (rolling mean/std and a W-cycle delta).
The **scope** of the window decides whether temporal locality is preserved:
- `['unit_nr']` → **partition-aware** (correct neighbours)
- `['rand_part']` → **partition-unaware** (scrambled neighbours; the ablation)

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # project root, so `import src` works

In [2]:
from src.spark_utils import get_spark
from src.ingest import ingest
from src.preprocess import preprocess
from src.windowing import add_windows
spark = get_spark()

In [3]:
df = ingest(spark, 'FD001')
df, keep = preprocess(df, 'FD001')
df_aw, feats_aw = add_windows(df, keep, ['unit_nr'])   # partition-aware
print(f'{len(keep)} sensors -> {len(feats_aw)} features after windowing')

15 sensors -> 60 features after windowing


Rolling features for one engine show the drift the detector will pick up:

In [4]:
c = keep[0]
df_aw.filter('unit_nr = 1').select('time_cycles', c, f'{c}_rm', f'{c}_dl')\
     .orderBy('time_cycles').show(10)

+-----------+--------------------+-------------------+------+
|time_cycles|                 s_2|             s_2_rm|s_2_dl|
+-----------+--------------------+-------------------+------+
|          1| -1.7216836613714954|-1.7216836613714954|   0.0|
|          2|  -1.061753971680569|-1.3917188165260321|   0.0|
|          3| -0.6617965839889196| -1.148411405680328|   0.0|
|          4| -0.6617965839889196| -1.026757700257476|   0.0|
|          5| -0.6218008452198001|-0.9457663292499408|   0.0|
|          6| -1.1617433186033679|-0.9817624941421786|   0.0|
|          7|-0.40182428198941555|-0.8989141781203553|   0.0|
|          8| -0.2418413269129376|-0.8167800717194281|   0.0|
|          9| -1.1217475798342484|-0.8506653503988526|   0.0|
|         10|   -1.94166022460188|-0.9597648378191554|   0.0|
+-----------+--------------------+-------------------+------+
only showing top 10 rows


In [5]:
spark.stop()